# 05 — Lr And Tree / 决策树

**Chapter 15 — File 5 of 6 / 第15章 — 第5个文件（共6个）**

---

## Summary / 总结

This script demonstrates **Adjust data types for categorical variables**.

本脚本演示 **Adjust data types for categorical variables**。

---
## Step 1 — Step 1

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

Ames = pd.read_csv("Ames.csv")

---
## Step 2 — Adjust data types for categorical variables

In [ ]:
for col in ["MSSubClass", "YrSold", "MoSold"]:
    Ames[col] = Ames[col].astype("object")

---
## Step 3 — Exclude "PID" and "SalePrice" from features and handle the "Electrical" column

In [ ]:
numeric_features = Ames.select_dtypes(include=["int64", "float64"]) \
                       .drop(columns=["PID", "SalePrice"]).columns
categorical_features = Ames.select_dtypes(include=["object"]).columns \
                           .difference(["Electrical"])
electrical_feature = ["Electrical"]

---
## Step 4 — Manually specify the categories for ordinal encoding according to the data dictionary

In [ ]:
ordinal_order = {

---
## Step 5 — Electrical system

In [ ]:
"Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],

---
## Step 6 — General shape of property

In [ ]:
"LotShape": ["IR3", "IR2", "IR1", "Reg"],

---
## Step 7 — Type of utilities available

In [ ]:
"Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],

---
## Step 8 — Slope of property

In [ ]:
"LandSlope": ["Sev", "Mod", "Gtl"],

---
## Step 9 — Evaluates the quality of the material on the exterior

In [ ]:
"ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 10 — Evaluates the present condition of the material on the exterior

In [ ]:
"ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 11 — Height of the basement

In [ ]:
"BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 12 — General condition of the basement

In [ ]:
"BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 13 — Walkout or garden level basement walls

In [ ]:
"BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],

---
## Step 14 — Quality of basement finished area

In [ ]:
"BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

---
## Step 15 — Quality of second basement finished area

In [ ]:
"BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

---
## Step 16 — Heating quality and condition

In [ ]:
"HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 17 — Kitchen quality

In [ ]:
"KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 18 — Home functionality

In [ ]:
"Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],

---
## Step 19 — Fireplace quality

In [ ]:
"FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 20 — Interior finish of the garage

In [ ]:
"GarageFinish": ["None", "Unf", "RFn", "Fin"],

---
## Step 21 — Garage quality

In [ ]:
"GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 22 — Garage condition

In [ ]:
"GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 23 — Paved driveway

In [ ]:
"PavedDrive": ["N", "P", "Y"],

---
## Step 24 — Pool quality

In [ ]:
"PoolQC": ["None", "Fa", "TA", "Gd", "Ex"],

---
## Step 25 — Fence quality

In [ ]:
"Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"]
}

---
## Step 26 — Extract list of ALL ordinal features from dictionary

In [ ]:
ordinal_features = list(ordinal_order.keys())

---
## Step 27 — List of ordinal features except Electrical

In [ ]:
ordinal_except_electrical = [feat for feat in ordinal_features if feat != "Electrical"]

---
## Step 28 — Define transformations for various feature types

In [ ]:
electrical_transformer = Pipeline(steps=[
    ("impute_electrical", SimpleImputer(strategy="most_frequent")),
    ("ordinal_electrical", OrdinalEncoder(categories=[ordinal_order["Electrical"]]))
])

numeric_transformer = Pipeline(steps=[
    ("impute_mean", SimpleImputer(strategy="mean"))
])

---
## Step 29 — Updated categorical imputer using SimpleImputer

In [ ]:
categorical_imputer = SimpleImputer(strategy="constant", fill_value="None")

ordinal_transformer = Pipeline([
    ("impute_ordinal", categorical_imputer),
    ("ordinal", OrdinalEncoder(categories=[ordinal_order[feat]
                                           for feat in ordinal_features
                                           if feat in ordinal_except_electrical]))
])

nominal_features = [feat for feat in categorical_features if feat not in ordinal_features]
categorical_transformer = Pipeline([
    ("impute_nominal", categorical_imputer),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

---
## Step 30 — Combined preprocessor for numeric, ordinal, nominal, and specific electrical data

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("electrical", electrical_transformer, ["Electrical"]),
        ("num", numeric_transformer, numeric_features),
        ("ordinal", ordinal_transformer, ordinal_except_electrical),
        ("nominal", categorical_transformer, nominal_features)
])

---
## Step 31 — "preprocessor" is already set up as your preprocessing pipeline

In [ ]:
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(random_state=42))
])

---
## Step 32 — Parameter distribution for RandomizedSearchCV

In [ ]:
param_dist = {

---
## Step 33 — Uniform distribution between 0.001 and 0.3

In [ ]:
"regressor__learning_rate": uniform(0.001, 0.299),

---
## Step 34 — Uniform distribution of integers from 100 to 500

In [ ]:
"regressor__n_estimators": randint(100, 501)
}

---
## Step 35 — Setup the RandomizedSearchCV

In [ ]:
random_search = RandomizedSearchCV(model_pipeline, param_distributions=param_dist,
                                   n_iter=50, cv=5, scoring="r2", verbose=1,
                                   random_state=42)

---
## Step 36 — Fit the RandomizedSearchCV to the data

In [ ]:
random_search.fit(Ames.drop(columns="SalePrice"), Ames["SalePrice"])

---
## Step 37 — Best parameters and best score from Random Search

In [ ]:
print("Best parameters (Random Search):", random_search.best_params_)
print("Best score (Random Search):", round((random_search.best_score_), 4))

---
## Learning Notes / 学习笔记

- **概念**: Adjust data types for categorical variables 是机器学习中的常用技术。  
  *Adjust data types for categorical variables is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Lr And Tree / 决策树
# Complete Code / 完整代码
# ===============================

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

Ames = pd.read_csv("Ames.csv")

# Adjust data types for categorical variables
for col in ["MSSubClass", "YrSold", "MoSold"]:
    Ames[col] = Ames[col].astype("object")

# Exclude "PID" and "SalePrice" from features and handle the "Electrical" column
numeric_features = Ames.select_dtypes(include=["int64", "float64"]) \
                       .drop(columns=["PID", "SalePrice"]).columns
categorical_features = Ames.select_dtypes(include=["object"]).columns \
                           .difference(["Electrical"])
electrical_feature = ["Electrical"]

# Manually specify the categories for ordinal encoding according to the data dictionary
ordinal_order = {
    # Electrical system
    "Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
    # General shape of property
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    # Type of utilities available
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    # Slope of property
    "LandSlope": ["Sev", "Mod", "Gtl"],
    # Evaluates the quality of the material on the exterior
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Evaluates the present condition of the material on the exterior
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Height of the basement
    "BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # General condition of the basement
    "BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Walkout or garden level basement walls
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],
    # Quality of basement finished area
    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # Quality of second basement finished area
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # Heating quality and condition
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Kitchen quality
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Home functionality
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    # Fireplace quality
    "FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Interior finish of the garage
    "GarageFinish": ["None", "Unf", "RFn", "Fin"],
    # Garage quality
    "GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Garage condition
    "GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Paved driveway
    "PavedDrive": ["N", "P", "Y"],
    # Pool quality
    "PoolQC": ["None", "Fa", "TA", "Gd", "Ex"],
    # Fence quality
    "Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"]
}

# Extract list of ALL ordinal features from dictionary
ordinal_features = list(ordinal_order.keys())

# List of ordinal features except Electrical
ordinal_except_electrical = [feat for feat in ordinal_features if feat != "Electrical"]

# Define transformations for various feature types
electrical_transformer = Pipeline(steps=[
    ("impute_electrical", SimpleImputer(strategy="most_frequent")),
    ("ordinal_electrical", OrdinalEncoder(categories=[ordinal_order["Electrical"]]))
])

numeric_transformer = Pipeline(steps=[
    ("impute_mean", SimpleImputer(strategy="mean"))
])

# Updated categorical imputer using SimpleImputer
categorical_imputer = SimpleImputer(strategy="constant", fill_value="None")

ordinal_transformer = Pipeline([
    ("impute_ordinal", categorical_imputer),
    ("ordinal", OrdinalEncoder(categories=[ordinal_order[feat]
                                           for feat in ordinal_features
                                           if feat in ordinal_except_electrical]))
])

nominal_features = [feat for feat in categorical_features if feat not in ordinal_features]
categorical_transformer = Pipeline([
    ("impute_nominal", categorical_imputer),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessor for numeric, ordinal, nominal, and specific electrical data
preprocessor = ColumnTransformer(
    transformers=[
        ("electrical", electrical_transformer, ["Electrical"]),
        ("num", numeric_transformer, numeric_features),
        ("ordinal", ordinal_transformer, ordinal_except_electrical),
        ("nominal", categorical_transformer, nominal_features)
])

# "preprocessor" is already set up as your preprocessing pipeline
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(random_state=42))
])


# Parameter distribution for RandomizedSearchCV
param_dist = {
    # Uniform distribution between 0.001 and 0.3
    "regressor__learning_rate": uniform(0.001, 0.299),
    # Uniform distribution of integers from 100 to 500
    "regressor__n_estimators": randint(100, 501)
}

# Setup the RandomizedSearchCV
random_search = RandomizedSearchCV(model_pipeline, param_distributions=param_dist,
                                   n_iter=50, cv=5, scoring="r2", verbose=1,
                                   random_state=42)

# Fit the RandomizedSearchCV to the data
random_search.fit(Ames.drop(columns="SalePrice"), Ames["SalePrice"])

# Best parameters and best score from Random Search
print("Best parameters (Random Search):", random_search.best_params_)
print("Best score (Random Search):", round((random_search.best_score_), 4))

---

➡️ **Next / 下一步**: File 6 of 6